In [15]:
import pandas as pd

from src.config_presets.tools.get_config import get_config

# metrics
from src.evaluation.metrics.classification import accuracy
from src.evaluation.metrics.calibration import ACE, ECE

In [16]:
# steps
# 1. read the predictions csv files for all models
# 2. compute accuracy and ACE scores for the validation set predictions
# 3. compute the mean metrics across all folds

In [31]:

def compute_and_print_metrics_per_fold(fold_folders, config):
    metrics_per_fold_list_dict = {}

    for folder in fold_folders:
        print(f"Processing folder: {folder}")
        predictions_file = f"{folder}/model_predictions.csv"
        df = pd.read_csv(predictions_file, sep=';')

        # keep only the validation preds
        df = df[df['Mode'] == 'val']

        # Extract endpoint names from column names
        endpoints = [col.replace('_pred', '') for col in df.columns if col.endswith('_pred')]

        # Create dictionaries for predictions and true labels per class
        predictions_per_class = {endpoint: df[f'{endpoint}_pred'].values for endpoint in endpoints}
        labels_per_class = {endpoint: df[f'{endpoint}_true'].values for endpoint in endpoints}


        
        # Compute accuracy and ACE scores for the validation set predictions
        for endpoint in endpoints:
            # mask out the missing labels (label == -1)
            mask = labels_per_class[endpoint] != -1
            
            predictions = predictions_per_class[endpoint][mask]
            true_labels = labels_per_class[endpoint][mask]

            acc = accuracy(config, true_labels, predictions)
            ace = ACE(config, true_labels, predictions)
            ece = ECE(config, true_labels, predictions)

            #print(f"Endpoint: {endpoint}, Accuracy: {acc:.4f}, ACE: {ace:.4f}, ECE: {ece:.4f}")
            metrics_per_fold_list_dict[folder] = metrics_per_fold_list_dict.get(folder, []) + [(endpoint, acc, ace, ece)]


    # Compute the mean metrics across all folds
    mean_metrics_per_endpoint = {}
    for endpoint in endpoints:
        acc_list = []
        ace_list = []
        ece_list = []
        
        for folder in fold_folders:
            for ep, acc, ace, ece in metrics_per_fold_list_dict[folder]:
                if ep == endpoint:
                    acc_list.append(acc)
                    ace_list.append(ace)
                    ece_list.append(ece)
        
        mean_acc = sum(acc_list) / len(acc_list)
        mean_ace = sum(ace_list) / len(ace_list)
        mean_ece = sum(ece_list) / len(ece_list)

        mean_metrics_per_endpoint[endpoint] = (mean_acc, mean_ace, mean_ece)

    # Print the mean metrics for each endpoint
    for endpoint, (mean_acc, mean_ace, mean_ece) in mean_metrics_per_endpoint.items():
        print(f"Endpoint: {endpoint}, Mean Accuracy: {mean_acc:.2f}, Mean ACE: {mean_ace:.2f}, Mean ECE: {mean_ece:.2f}")
    

In [ ]:
# MULTI TOX

root_dir = "/home/macraedc/results/MICCAI/Model_reproductions/Multi_tox"
config = get_config('Daniel/MICCAI/Multi_tox')

fold_folders = [f"{root_dir}/KFold{i}" for i in range(1, 6)]

compute_and_print_metrics_per_fold(fold_folders, config)

src/config_presets/Base_config.yaml
src/config_presets/Daniel/MICCAI/Multi_tox.yaml
Processing folder: /home/macraedc/results/MICCAI/Model_reproductions/Multi_tox/KFold1
Processing folder: /home/macraedc/results/MICCAI/Model_reproductions/Multi_tox/KFold2
Processing folder: /home/macraedc/results/MICCAI/Model_reproductions/Multi_tox/KFold3
Processing folder: /home/macraedc/results/MICCAI/Model_reproductions/Multi_tox/KFold4
Processing folder: /home/macraedc/results/MICCAI/Model_reproductions/Multi_tox/KFold5
Endpoint: Aspiration_M06, Mean Accuracy: 0.72, Mean ACE: 0.09, Mean ECE: 0.28
Endpoint: Dysphagia_M06, Mean Accuracy: 0.74, Mean ACE: 0.06, Mean ECE: 0.26
Endpoint: Sticky_M06, Mean Accuracy: 0.64, Mean ACE: 0.07, Mean ECE: 0.36
Endpoint: Taste_M06, Mean Accuracy: 0.58, Mean ACE: 0.07, Mean ECE: 0.42
Endpoint: Xerostomia_M06, Mean Accuracy: 0.67, Mean ACE: 0.06, Mean ECE: 0.33


In [ ]:
# SINGLE TOX

config = get_config('Daniel/MICCAI/Multi_tox')

endpoints_list = ['Aspiration_M06', 'Dysphagia_M06', 'Sticky_M06', 'Taste_M06', 'Xerostomia_M06']

for endpoint in endpoints_list:
    root_dir = f"/home/macraedc/results/MICCAI/ST_models/{endpoint}"

    fold_folders = [f"{root_dir}/KFold{i}" for i in range(1, 6)]

    compute_and_print_metrics_per_fold(fold_folders, config)

src/config_presets/Base_config.yaml
src/config_presets/Daniel/MICCAI/Multi_tox.yaml
Processing folder: /home/macraedc/results/MICCAI/ST_models/Aspiration_M06/KFold1
Processing folder: /home/macraedc/results/MICCAI/ST_models/Aspiration_M06/KFold2
Processing folder: /home/macraedc/results/MICCAI/ST_models/Aspiration_M06/KFold3
Processing folder: /home/macraedc/results/MICCAI/ST_models/Aspiration_M06/KFold4
Processing folder: /home/macraedc/results/MICCAI/ST_models/Aspiration_M06/KFold5
Endpoint: Aspiration_M06, Mean Accuracy: 0.71, Mean ACE: 0.08, Mean ECE: 0.29
Processing folder: /home/macraedc/results/MICCAI/ST_models/Dysphagia_M06/KFold1
Processing folder: /home/macraedc/results/MICCAI/ST_models/Dysphagia_M06/KFold2
Processing folder: /home/macraedc/results/MICCAI/ST_models/Dysphagia_M06/KFold3
Processing folder: /home/macraedc/results/MICCAI/ST_models/Dysphagia_M06/KFold4
Processing folder: /home/macraedc/results/MICCAI/ST_models/Dysphagia_M06/KFold5
Endpoint: Dysphagia_M06, Mean Acc